给定如下的积分函数
$$
f(x) = \int_{0}^{1} g(t, x) \, \mathrm{d}t = \int_{0}^{1} [t + \sin(x t^2)] \, \mathrm{d}t.
$$

In [1]:
import math
import numpy as np
import time

def g(t, x):
    return t + math.sin(x * t * t)
def trapezoidal_rule(a, b, n, x):
    """计算复合梯形公式"""
    h = (b - a) / n
    t = np.linspace(a, b, n + 1)
    y = [g(ti, x) for ti in t]
    result = (h / 2) * (y[0] + 2 * sum(y[1:-1]) + y[-1])
    return result

## 1
当 $x=1$, 使用复化梯形积分计算该积分, 并验证误差的二阶精确性。这里积分准确值 $f(1)$ 未知,
- (a) 用非常小的区间长度 $\Delta t = \frac{1}{n}$ 下的数值积分来代替 $f(1)$。计算在 $n=n_0, 2n_0, 4n_0, \dots$, 下的积分误差 $E_n(g) = I_n(g(\cdot, 1)) - f(1)$, 并基于 $\frac{|E_n(g)|}{|E_{2n}(g)|}$ 计算收敛阶

In [2]:
# 设置参数
x_val = 1
a = 0
b = 1

# (a) 计算参考 "真值"
# 选择一个足够大的 n 来获得高精度的近似值
n_true = 10 * 2**10  # n = 10240
I_true = trapezoidal_rule(a, b, n_true, x_val)
print(f"参考 '真值' I_true (使用 n={n_true}): {I_true:.12f}")
print("-" * 30)

# 计算一系列 n 值下的积分和误差
n0 = 1
num_levels = 10
n_values = [n0 * (2**i) for i in range(num_levels)]

results_a = []
errors_a = []
ratios_a = []

print("Part 1(a): 使用参考真值验证收敛阶")
print(f"{'n':>5s} {'I_n':>15s} {'Error E_n':>15s} {'|E_n|/|E_2n|':>15s}")
print("-" * 55)

# 计算 I_n 和 E_n
for n in n_values:
    I_n = trapezoidal_rule(a, b, n, x_val)
    E_n = I_n - I_true
    results_a.append(I_n)
    errors_a.append(E_n)

# 计算误差比率 |E_n| / |E_{2n}|
for i in range(num_levels - 1):
    if abs(errors_a[i+1]) > 1e-15: # 避免除以零
        ratio = abs(errors_a[i]) / abs(errors_a[i+1])
    else:
        ratio = np.nan # 无法计算
    ratios_a.append(ratio)
    print(f"{n_values[i]:>5d} {results_a[i]:>15.10f} {errors_a[i]:>15.3e} {ratio:>15.5f}" if i < len(ratios_a) else f"{n_values[i]:>5d} {results_a[i]:>15.10f} {errors_a[i]:>15.3e} {'N/A':>15s}")

# 打印最后一个 n 的结果
print(f"{n_values[-1]:>5d} {results_a[-1]:>15.10f} {errors_a[-1]:>15.3e} {'N/A':>15s}")

参考 '真值' I_true (使用 n=10240): 0.810268302582
------------------------------
Part 1(a): 使用参考真值验证收敛阶
    n             I_n       Error E_n    |E_n|/|E_2n|
-------------------------------------------------------
    1    0.9207354924       1.105e-01         4.64120
    2    0.8340697258       2.380e-02         4.17052
    4    0.8159753608       5.707e-03         4.04201
    8    0.8116802395       1.412e-03         4.01045
   16    0.8106203668       3.521e-04         4.00264
   32    0.8103562607       8.796e-05         4.00077
   64    0.8102902879       2.199e-05         4.00063
  128    0.8102737980       5.495e-06         4.00192
  256    0.8102696758       1.373e-06         4.00753
  512    0.8102686452       3.427e-07             N/A


- (b) 利用如下相对误差公式计算
$$
\frac{|I_n(g) - I_{2n}(g)|}{|I_{2n}(g) - I_{4n}(g)|}.
$$

In [3]:
# (b) 计算相对误差比率
ratios_b = []
print(f"{'n':>5s} {'I_n':>15s} {'|I_n - I_2n|':>15s} {'Ratio (b)':>15s}")
print("-" * 55)

# 计算比率 |I_n - I_{2n}| / |I_{2n} - I_{4n}|
for i in range(num_levels - 2):
    diff1 = results_a[i] - results_a[i+1]
    diff2 = results_a[i+1] - results_a[i+2]
    if abs(diff2) > 1e-15: # 避免除以零
        ratio = abs(diff1) / abs(diff2)
    else:
        ratio = np.nan # 无法计算
    ratios_b.append(ratio)
    print(f"{n_values[i]:>5d} {results_a[i]:>15.10f} {abs(diff1):>15.3e} {ratio:>15.5f}" if i < len(ratios_b) else f"{n_values[i]:>5d} {results_a[i]:>15.10f} {abs(diff1):>15.3e} {'N/A':>15s}")

# 打印最后两个 n 的相关信息
if num_levels >= 2:
    diff_last = results_a[-2] - results_a[-1]
    print(f"{n_values[-2]:>5d} {results_a[-2]:>15.10f} {abs(diff_last):>15.3e} {'N/A':>15s}")
if num_levels >= 1:
    print(f"{n_values[-1]:>5d} {results_a[-1]:>15.10f} {'N/A':>15s} {'N/A':>15s}")

    n             I_n    |I_n - I_2n|       Ratio (b)
-------------------------------------------------------
    1    0.9207354924       8.667e-02         4.78966
    2    0.8340697258       1.809e-02         4.21277
    4    0.8159753608       4.295e-03         4.05249
    8    0.8116802395       1.060e-03         4.01306
   16    0.8106203668       2.641e-04         4.00326
   32    0.8103562607       6.597e-05         4.00081
   64    0.8102902879       1.649e-05         4.00020
  128    0.8102737980       4.122e-06         4.00005
  256    0.8102696758       1.031e-06             N/A
  512    0.8102686452             N/A             N/A


## 2
当 $x=1$, 使用复化Simpson积分计算积分, 并验证误差的四阶精确性。


In [4]:
def composite_simpson(func, a, b, N):
    """
    使用复化 Simpson 法则计算定积分。

    参数:
        func (callable): 被积函数，接受一个参数 t。
        a (float): 积分下限。
        b (float): 积分上限。
        N (int): 子区间的数量 (必须是偶数)。

    返回:
        float: 积分的近似值。
    """
    if N % 2 != 0:
        raise ValueError("子区间数量 N 必须是偶数。")

    h = (b - a) / N
    integral_sum = func(a) + func(b) # 加上端点值 f(t_0) + f(t_N)

    # 加上奇数点的项 (系数为 4)
    for i in range(1, N, 2):
        t_i = a + i * h
        integral_sum += 4 * func(t_i)

    # 加上偶数点的项 (系数为 2)
    for i in range(2, N - 1, 2):
        t_i = a + i * h
        integral_sum += 2 * func(t_i)

    integral = (h / 3) * integral_sum
    return integral

In [5]:
a = 0.0
b = 1.0
x_val = 1.0 # 计算 f(1)

# 验证四阶精度
N_values = [10, 20, 40, 80, 160, 320] # 一系列 N 值，每次加倍
results = []

# 计算一个高精度的参考值
N_ref = 10000
I_ref = composite_simpson(lambda t: g(t, x_val), a, b, N_ref)
print(f"\n使用 N={N_ref} 计算的参考积分值 I_ref = {I_ref:.15f}")

print("\n验证复化 Simpson 法则的四阶精度:")
print("-" * 60)
print(f"{'N':>6s} {'Approximation (S_N)':>22s} {'Est. Error |I_ref - S_N|':>25s} {'Ratio E_N / E_{N/2}':>20s}")
print("-" * 60)

last_error = None
for N in N_values:
    S_N = composite_simpson(lambda t: g(t, x_val), a, b, N)
    error = abs(I_ref - S_N)
    ratio = None
    if last_error is not None and error > 1e-16: # 避免除以非常小的数或零
        ratio = error / last_error
        results.append({'N': N, 'S_N': S_N, 'Error': error, 'Ratio': ratio})
        print(f"{N:6d} {S_N:22.15f} {error:25.4e} {ratio:20.6f}")
    else:
        results.append({'N': N, 'S_N': S_N, 'Error': error, 'Ratio': None})
        print(f"{N:6d} {S_N:22.15f} {error:25.4e} {'N/A':>20s}")

    last_error = error

print("-" * 60)
print(f"预期的比率趋近于 1/16 = {1/16:.6f}")


使用 N=10000 计算的参考积分值 I_ref = 0.810268301723382

验证复化 Simpson 法则的四阶精度:
------------------------------------------------------------
     N    Approximation (S_N)  Est. Error |I_ref - S_N|  Ratio E_N / E_{N/2}
------------------------------------------------------------
    10      0.810260234433221                8.0673e-06                  N/A
    20      0.810267800132075                5.0159e-07             0.062176
    40      0.810268270415785                3.1308e-08             0.062417
    80      0.810268299767314                1.9561e-09             0.062479
   160      0.810268301601137                1.2224e-10             0.062495
   320      0.810268301715741                7.6408e-12             0.062504
------------------------------------------------------------
预期的比率趋近于 1/16 = 0.062500


## 3
在 1, 2 中, 当 $n$ 很小或者 $n$ 很大时是否有收敛阶? (hint: $n$ 很小时, 没有对应的收敛阶是因为分辨率不足; $n$ 很大时, 也观察不到对应收敛阶的原因是什么?)

1.  **当 $n$ 很小时:**
    *   **原因:** 理论误差公式 $E_n \approx C h^p$ (其中 $p=2$ 或 $p=4$) 是通过泰勒展开推导的，并忽略了高阶项。当 $n$ 很小，步长 $h$ 就很大。对于较大的 $h$，那些被忽略的高阶项（例如，对于梯形法则是 $O(h^3)$ 或 $O(h^4)$ 的项，对于辛普森法则是 $O(h^5)$ 或 $O(h^6)$ 的项）可能仍然相当显著，不能忽略。
2.  **当 $n$ 很大时:**
    *   **原因:** 理论上，随着 $n$ 增大，$h$ 变小，截断误差 $E_n \approx C h^p$ 应该持续减小，收敛阶应该越来越明显。然而，在实际计算中，我们使用的是有限精度的浮点数（例如 `float64`）。当 $n$ 变得非常大时：舍入误差累积, 舍入误差超过截断误差


## 4
在 $x$ 很大时, $t + \sin(xt^2)$ 在区间 $[0, 1]$ 上会出现很多的震荡。为了看到准确的收敛阶, 复化积分的区间长度 $\Delta t = \frac{1}{n}$ 则要求足够小。分别考虑 $x=100, 1000, 10000$, 并利用 1(b) 中求误差收敛阶公式, 探索需要对应的 $n$ 多大时, 才能看到清晰的收敛阶?


In [6]:
# --- 实验设置 ---
x_values = [100, 1000, 10000]
max_k = 20
n_values = [2**k for k in range(max_k + 1)] # n = 1, 2, 4, ..., 2^max_k

# --- 对每个 x 值运行实验 ---
for x in x_values:
    print(f"\n===== 分析 x = {x} =====")
    print(f"{'n':>10s} {'I_n':>20s} {'|I_n - I_2n|':>18s} {'Ratio (|I_n-I_2n|/|I_2n-I_4n|)':>30s}")
    print("-" * 85)

    results = []

    # 计算所有 n 值的 I_n
    for i, n in enumerate(n_values):
        integral_val = trapezoidal_rule(0, 1, n, x)
        results.append(integral_val)

    # 计算差值和比率
    ratios = []
    diffs = []
    if len(results) > 1:
        for i in range(len(results) - 1):
            diff = results[i] - results[i+1]
            diffs.append(diff)

    if len(diffs) > 1:
        for i in range(len(diffs) - 1):
            diff1 = diffs[i]
            diff2 = diffs[i+1]
            # 增加对 diff1 和 diff2 的检查，避免它们过小导致比率无意义
            if abs(diff2) > 1e-16 and abs(diff1) > 1e-16 : # 避免除以零或非常小的数
                ratio = abs(diff1) / abs(diff2)
                # 如果比率非常大或非常小，可能还未进入渐近区
                if ratio > 1000 or ratio < 0.01:
                    ratio = np.nan # 标记为无效比率
            else:
                ratio = np.nan # 无法计算或无意义
            ratios.append(ratio)

    # 打印结果表格
    for i in range(len(n_values)):
        n_val = n_values[i]
        I_n_val = results[i]

        if i < len(diffs):
            # 检查差值是否为 NaN 或 Inf
            if np.isnan(diffs[i]) or np.isinf(diffs[i]):
                 diff_val_str = f"{'Invalid Diff':>18s}"
            else:
                 diff_val_str = f"{abs(diffs[i]):>18.3e}"
        else:
            diff_val_str = f"{'N/A':>18s}" # 最后一个 n 没有差值

        if i < len(ratios):
             # 检查比率是否为 NaN 或 Inf
             if np.isnan(ratios[i]) or np.isinf(ratios[i]):
                  ratio_val_str = f"{'Invalid Ratio':>30s}"
             else:
                  ratio_val_str = f"{ratios[i]:>30.5f}"
        else:
             ratio_val_str = f"{'N/A':>30s}" # 最后两个 n 没有比率

        print(f"{n_val:>10d} {I_n_val:>20.10f} {diff_val_str} {ratio_val_str}")



===== 分析 x = 100 =====
         n                  I_n       |I_n - I_2n| Ratio (|I_n-I_2n|/|I_2n-I_4n|)
-------------------------------------------------------------------------------------
         1         0.2468171794          6.042e-02                        4.15883
         2         0.3072327147          1.453e-02                        0.02522
         4         0.3217597503          5.759e-01                        2.83720
         8         0.8976610812          2.030e-01                        1.89128
        16         0.6946785466          1.073e-01                        4.33765
        32         0.5873530191          2.474e-02                        7.43378
        64         0.5626102313          3.328e-03                        4.80174
       128         0.5592818038          6.932e-04                        4.16123
       256         0.5585886321          1.666e-04                        4.03841
       512         0.5584220534          4.125e-05                    


## 5
考虑 $x=1, 100$ 时的 Romberg 积分, 其中精度控制 $\epsilon = 10^{-12}$。

In [7]:
def romberg_integration(a, b, x, eps=1e-12, max_iter=20):
    """Romberg 积分"""
    R = np.zeros((max_iter, max_iter))
    for i in range(max_iter):
        # 计算 R(i, 0)，即 2^i 个子区间的梯形公式
        n = 2 ** i
        R[i, 0] = trapezoidal_rule(a, b, n, x)

        # 计算 R(i, j) for j >= 1
        for j in range(1, i + 1):
            R[i, j] = R[i, j-1] + (R[i, j-1] - R[i-1, j-1]) / (4**j - 1)

        # 检查收敛性
        if i > 0 and abs(R[i, i] - R[i-1, i-1]) < eps:
            return R[i, i], i

    # 如果未收敛，返回最后的结果并警告
    print(f"Warning: Did not converge within {max_iter} iterations for x={x}")
    return R[max_iter-1, max_iter-1], max_iter - 1

# 参数设置
a, b = 0, 1  # 积分区间 [0, 1]
epsilon = 1e-12  # 精度
x_values = [1, 100]  # x = 1 和 x = 100

for x in x_values:
    result, iterations = romberg_integration(a, b, x, epsilon)
    print(f"x = {x}:")
    print(f"  积分值: {result:.12e}")
    print(f"  迭代次数: {iterations}")

x = 1:
  积分值: 8.102683017234e-01
  迭代次数: 6
x = 100:
  积分值: 5.583670899930e-01
  迭代次数: 12


## 6

编程实现二重积分的复化Simpson积分, 并计算如下函数,
$$
\displaystyle \int_0^1 \int_0^3 e^{x+y} \, \mathrm{d}x \, \mathrm{d}y.
$$

$$
 \int_0^{10} \int_0^1 [t + \sin(x t^2)] \, \mathrm{d}t \, \mathrm{d}x.
$$
减小 $x$ 与 $y$ 方向上的区间长度, 是否可以观察到误差减小?

In [8]:
import numpy as np

def composite_simpson_double(f, a, b, c, d, nx, ny):
    """
    计算二重积分 ∫[c,d] ∫[a,b] f(x,y) dx dy 使用复化Simpson法则.

    参数:
    f: 被积函数 f(x, y).
    a, b: x 的积分下限和上限.
    c, d: y 的积分下限和上限.
    nx: x 方向上的子区间数量 (必须是偶数).
    ny: y 方向上的子区间数量 (必须是偶数).

    返回:
    积分的近似值.
    """
    if nx % 2 != 0 or ny % 2 != 0:
        raise ValueError("子区间数量 nx 和 ny 必须是偶数")
    if nx <= 0 or ny <= 0:
        raise ValueError("子区间数量 nx 和 ny 必须是正数")

    hx = (b - a) / nx
    hy = (d - c) / ny

    integral_sum = 0.0

    for j in range(ny + 1):  # 遍历 y 方向 (外层)
        y = c + j * hy
        # 确定 y 方向的权重 Wy
        if j == 0 or j == ny:
            Wy = 1
        elif j % 2 != 0: # 奇数索引
            Wy = 4
        else: # 偶数索引 (非边界)
            Wy = 2

        for i in range(nx + 1): # 遍历 x 方向 (内层)
            x = a + i * hx
            # 确定 x 方向的权重 Wx
            if i == 0 or i == nx:
                Wx = 1
            elif i % 2 != 0: # 奇数索引
                Wx = 4
            else: # 偶数索引 (非边界)
                Wx = 2

            # 累加带权重的函数值
            integral_sum += Wx * Wy * f(x, y)

    # 乘以系数 hx*hy/9
    result = (hx * hy / 9.0) * integral_sum
    return result

In [9]:
# --- 问题 1 ---
def f1(x, y):
  """被积函数 e^(x+y)"""
  return np.exp(x + y)

# 积分限
a1, b1 = 0, 3  # x 的范围
c1, d1 = 0, 1  # y 的范围

# 计算精确值用于比较
exact_value1 = (np.exp(3) - 1) * (np.exp(1) - 1)
print(f"精确值: {exact_value1:.12f}")

# 使用不同的子区间数量计算并观察误差
print("\n使用复化 Simpson 法计算:")
errors1 = []
results1 = []
n_values = [4, 8, 16, 32, 64] # 必须是偶数

for n in n_values:
    nx = n  # x 方向子区间数
    ny = n  # y 方向子区间数 (可以不同, 但这里取相同方便观察)
    result = composite_simpson_double(f1, a1, b1, c1, d1, nx, ny)
    error = abs(result - exact_value1)
    results1.append(result)
    errors1.append(error)
    print(f"nx={nx}, ny={ny}: 近似值 = {result:.12f}, 误差 = {error:.6e}")

精确值: 32.794331281498

使用复化 Simpson 法计算:
nx=4, ny=4: 近似值 = 32.849040505071, 误差 = 5.470922e-02
nx=8, ny=8: 近似值 = 32.797919138392, 误差 = 3.587857e-03
nx=16, ny=16: 近似值 = 32.794558302061, 误差 = 2.270206e-04
nx=32, ny=32: 近似值 = 32.794345514316, 误差 = 1.423282e-05
nx=64, ny=64: 近似值 = 32.794332171739, 误差 = 8.902416e-07


In [10]:
# --- 问题 2 ---
def f2(t, x):
  """被积函数 t + sin(x * t^2)"""
  # 注意函数参数顺序: 第一个参数是内层积分变量(t), 第二个是外层(x)
  return t + np.sin(x * t**2)

# 积分限
a2, b2 = 0, 1   # t 的范围
c2, d2 = 0, 10  # x 的范围

results2 = []
n_values2 = [10, 20, 40, 80, 160] # 必须是偶数

for n in n_values2:
    nt = n  # t 方向子区间数 (内层)
    nx_outer = n  # x 方向子区间数 (外层)
    result = composite_simpson_double(f2, a2, b2, c2, d2, nt, nx_outer)
    results2.append(result)
    print(f"nt={nt}, nx_outer={nx_outer}: 近似值 = {result:.12f}")
    if len(results2) > 1:
        diff = abs(results2[-1] - results2[-2])
        print(f"  与上一个结果的差值: {diff:.6e}")

nt=10, nx_outer=10: 近似值 = 7.988562857145
nt=20, nx_outer=20: 近似值 = 7.984053682558
  与上一个结果的差值: 4.509175e-03
nt=40, nx_outer=40: 近似值 = 7.983807967092
  与上一个结果的差值: 2.457155e-04
nt=80, nx_outer=80: 近似值 = 7.983793490300
  与上一个结果的差值: 1.447679e-05
nt=160, nx_outer=160: 近似值 = 7.983792598935
  与上一个结果的差值: 8.913657e-07
